# Sommeil EOG — Reprendre CNN NPU (checkpoint local)

Continue l'entrainement **CNN 1D NPU** interrompu sur PC (`sleep_model_cnn_best.keras`).

## Datasets a ajouter (+ Add data)

1. **Corpus** : `sommeil-eog-corpus-preprocessed` (3 fichiers npz/json)
2. **Checkpoint** : uploadez `sleep_model_cnn_best.keras` depuis `models/` sur votre PC
   - soit dans le **meme dataset** (4 fichiers)
   - soit dans un **2e dataset** separe

Puis **Run All**. GPU ON, Internet OFF.

In [ ]:
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score
from sklearn.utils import class_weight
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import (
    BatchNormalization, Conv1D, Dense, Dropout,
    GlobalAveragePooling1D, MaxPooling1D,
)
from tensorflow.keras.models import Sequential

SEED = 42
EPOCHS_TOTAL = 30
INITIAL_EPOCH = 7   # epochs deja faites sur PC (ajuster si besoin)
BATCH_SIZE = 64
PATIENCE = 7
STAGE_NAMES = ["W", "N1", "N2", "N3", "REM"]
CORPUS_FILES = ("sleep_edf_corpus.npz", "sleep_edf_corpus_meta.json", "subject_split.json")
CHECKPOINT_NAME = "sleep_model_cnn_best.keras"

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def find_file(name):
    root = Path("/kaggle/input")
    hits = sorted(root.glob(f"**/{name}"))
    if not hits:
        raise FileNotFoundError(f"{name} introuvable sous /kaggle/input — ajoutez le dataset.")
    return hits[0]


def corpus_dir_from_npz(npz_path):
    d = npz_path.parent
    if all((d / f).exists() for f in CORPUS_FILES):
        return d
    raise FileNotFoundError("Dataset corpus incomplet (3 fichiers requis).")


NPZ_PATH = find_file("sleep_edf_corpus.npz")
INPUT_DIR = corpus_dir_from_npz(NPZ_PATH)
CHECKPOINT_PATH = find_file(CHECKPOINT_NAME)

tf.random.set_seed(SEED)
np.random.seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

print("Corpus  :", INPUT_DIR)
print("Checkpoint :", CHECKPOINT_PATH)
print("GPU :", tf.config.list_physical_devices("GPU"))

In [ ]:
def masks_from_manifest(subject_idx, subject_names, manifest):
    name_to_id = {name: i for i, name in enumerate(subject_names)}
    def _mask(names):
        ids = {name_to_id[n] for n in names if n in name_to_id}
        return np.isin(subject_idx, list(ids))
    return _mask(manifest["train_subjects"]), _mask(manifest["val_subjects"]), _mask(manifest["test_subjects"])


class F1MacroCallback(tf.keras.callbacks.Callback):
    def __init__(self, X_val, y_val, patience=PATIENCE):
        super().__init__()
        self.X_val, self.y_val = X_val, y_val
        self.patience, self.best_f1, self.wait = patience, -1.0, 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_pred = np.argmax(self.model.predict(self.X_val, verbose=0), axis=1)
        macro = float(f1_score(self.y_val, y_pred, average="macro", zero_division=0))
        logs["val_f1_macro"] = macro
        print(f"  -> val_f1_macro = {macro:.4f}")
        if macro > self.best_f1 + 1e-4:
            self.best_f1, self.wait = macro, 0
        else:
            self.wait += 1
        if self.wait >= self.patience:
            self.model.stop_training = True


data = np.load(NPZ_PATH)
X, y = data["X"].astype(np.float32), data["y"].astype(np.int32)
subject_idx = data["subject_idx"].astype(np.int32)
with open(INPUT_DIR / "sleep_edf_corpus_meta.json", encoding="utf-8") as f:
    subject_names = json.load(f)["subject_names"]
with open(INPUT_DIR / "subject_split.json", encoding="utf-8") as f:
    manifest = json.load(f)
train_mask, val_mask, _ = masks_from_manifest(subject_idx, subject_names, manifest)
X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
del data

class_weights = dict(enumerate(class_weight.compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)))
print(f"Train {len(y_train):,} epoques · Val {len(y_val):,} epoques")

In [ ]:
model = tf.keras.models.load_model(CHECKPOINT_PATH)
model.summary()

best_path = WORK_DIR / "sleep_model_cnn_best.keras"
callbacks = [
    F1MacroCallback(X_val, y_val),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(best_path), monitor="val_f1_macro", mode="max", save_best_only=True, verbose=1),
]

remaining = max(1, EPOCHS_TOTAL - INITIAL_EPOCH)
print(f"Reprise epoch {INITIAL_EPOCH + 1} -> {EPOCHS_TOTAL} (max {remaining} epochs)")
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    initial_epoch=INITIAL_EPOCH,
    epochs=EPOCHS_TOTAL,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=callbacks,
    shuffle=True,
    verbose=2,
)

if best_path.exists():
    model = tf.keras.models.load_model(best_path)
model.save(WORK_DIR / "sleep_model_cnn.keras")
print("Termine — telechargez sleep_model_cnn.keras et sleep_model_cnn_best.keras")